<a href="https://colab.research.google.com/github/FariaZulfiqar/Supply-Chain-Analytics-Portfolio-Faria-Zulfiqar-IBA-Fall2026/blob/main/00-Initial-Working/Intial-test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supply Chain Analytics: Top Vendors and Procurement Metrics

This notebook analyzes procurement data from the `Chenab_Motors_Procurement_Practice_Dataset.xlsx` file to identify key vendors and calculate important procurement metrics.

## Step 1: Data Loading

The first step is to load the Excel dataset into a pandas DataFrame. Based on prior inspection, the actual tabular data starts at row 20 (0-indexed, meaning `header=19`). This accounts for the introductory metadata and data dictionary in the Excel file.

In [24]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the file path for the uploaded Excel dataset
file_path = '/content/Chenab_Motors_Procurement_Practice_Dataset.xlsx'

# Load the dataset into a pandas DataFrame, specifying the header row
df = pd.read_excel(file_path, header=19)
print(f"Successfully loaded data from: {file_path} with header at row 20 (0-indexed 19).")

# Display the first few rows and information of the loaded DataFrame for initial inspection
print("\nFirst 5 rows of the loaded dataset:")
display(df.head())
print("\nDataFrame Info of the loaded dataset:")
df.info()

Successfully loaded data from: /content/Chenab_Motors_Procurement_Practice_Dataset.xlsx with header at row 20 (0-indexed 19).

First 5 rows of the loaded dataset:


,PO Number,Purchase order identifier (may repeat across multiple line items),Unnamed: 2
0,PO Line Item,Line number within the PO,NaN
1,Vendor ID / Vendor,Supplier identifier and name,NaN
2,Plant,Manufacturing site the PO was raised for,NaN
3,Department / Cost Center,Requesting department and its cost center code,NaN
4,Requisitioner,Employee who raised the requisition,NaN



DataFrame Info of the loaded dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 3 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   PO Number                                                          14 non-null     object 
 1   Purchase order identifier (may repeat across multiple line items)  14 non-null     object 
 2   Unnamed: 2                                                         0 non-null      float64
dtypes: float64(1), object(2)
memory usage: 468.0+ bytes


## Step 2: Data Inspection and Cleaning

Next, we inspect the data for its structure, data types, and any missing values. We will then clean the data by converting the 'PO Value (PKR)' column to a numeric type and handling missing values in 'PO Value (PKR)' and 'Vendor ID / Vendor'.

In [25]:
# Check for the number of missing values in each column
print("\nMissing values per column before cleaning:")
display(df.isnull().sum())

# --- Data Cleaning Steps ---

# 1. Convert 'PO Value (PKR)' to a numeric type.
#    'errors=\'coerce\'' will turn non-numeric values into NaN.
#    This column represents the procurement value.
if 'PO Value (PKR)' in df.columns:
    df['PO Value (PKR)'] = pd.to_numeric(df['PO Value (PKR)'], errors='coerce')
    # Fill any NaN values in 'PO Value (PKR)' with 0, as a missing amount implies no contribution to procurement.
    df['PO Value (PKR)'].fillna(0, inplace=True)
    print("\n'PO Value (PKR)' column cleaned and missing values filled with 0.")
else:
    print("Warning: 'PO Value (PKR)' column not found. Please ensure the column name is correct for procurement value.")

# 2. Handle missing values in 'Vendor ID / Vendor' by filling them with 'Unknown Vendor'.
if 'Vendor ID / Vendor' in df.columns:
    df['Vendor ID / Vendor'].fillna('Unknown Vendor', inplace=True)
    print("'Vendor ID / Vendor' column cleaned and missing values filled with 'Unknown Vendor'.")
else:
    print("Warning: 'Vendor ID / Vendor' column not found. Please ensure the column name is correct for vendors.")

# Re-check missing values after cleaning
print("\nMissing values per column after cleaning:")
display(df.isnull().sum())


Missing values per column before cleaning:


,0
PO Number,0
Purchase order identifier (may repeat across multiple line items),0
Unnamed: 2,14



Missing values per column after cleaning:


,0
PO Number,0
Purchase order identifier (may repeat across multiple line items),0
Unnamed: 2,14


## Step 3: Calculate Key Procurement Metrics

Here, we calculate several important procurement metrics:
-   **Total PO Spend**: The sum of all 'PO Value (PKR)'.
-   **Total Number of POs**: The count of unique 'PO Number' entries.
-   **Total Number of Vendors**: The count of unique 'Vendor ID / Vendor' entries.
-   **Average PO Value**: Total PO Spend divided by the Total Number of POs.
-   **Average Line Item Value**: Total PO Spend divided by the total number of line items (rows in the DataFrame).

In [29]:
# Calculate Total PO Spend
total_po_spend = df['PO Value (PKR)'].sum()

# Calculate Total Number of POs (using unique 'PO Number')
total_num_pos = df['PO Number'].nunique()

# Calculate Total Number of Vendors (using unique 'Vendor ID / Vendor')
total_num_vendors = df['Vendor ID / Vendor'].nunique()

# Calculate Average PO Value
# Ensure total_num_pos is not zero to avoid division by zero error
average_po_value = total_po_spend / total_num_pos if total_num_pos > 0 else 0

# Calculate Average Line Item Value (Total PO Spend divided by total number of rows)
total_line_items = len(df)
average_line_item_value = total_po_spend / total_line_items if total_line_items > 0 else 0

# Display the calculated metrics
print(f"Total PO Spend: PKR {total_po_spend:,.2f}")
print(f"Total Number of POs: {total_num_pos}")
print(f"Total Number of Vendors: {total_num_vendors}")
print(f"Average PO Value: PKR {average_po_value:,.2f}")
print(f"Average Line Item Value: PKR {average_line_item_value:,.2f}")

KeyError: 'PO Value (PKR)'

## Step 4: Identify Top 3 Vendors

To find the top 3 vendors, we will group the data by 'Vendor ID / Vendor' and sum their respective 'PO Value (PKR)'. The results will then be sorted in descending order to easily pick out the top performers.

In [30]:
# Group the DataFrame by 'Vendor ID / Vendor' and sum the 'PO Value (PKR)' for each vendor
vendor_procurement = df.groupby('Vendor ID / Vendor')['PO Value (PKR)'].sum().reset_index()

# Sort the vendors by their total procurement amount in descending order
vendor_procurement_sorted = vendor_procurement.sort_values(by='PO Value (PKR)', ascending=False)

# Select the top 3 vendors from the sorted list
top_3_vendors = vendor_procurement_sorted.head(3)

print("Top 3 Vendors by Total Procurement Amount:")
display(top_3_vendors)

KeyError: 'Vendor ID / Vendor'

## Step 5: Visualization: Top 3 Vendors by Procurement Amount

A bar chart provides a clear visual comparison of the total procurement amounts from the top 3 vendors, making it easy to understand their relative contributions.

In [31]:
# Set a larger figure size for better readability of the plot
plt.figure(figsize=(12, 7))

# Create a bar plot using seaborn, showing 'Vendor ID / Vendor' on the x-axis and 'PO Value (PKR)' on the y-axis
sns.barplot(x='Vendor ID / Vendor', y='PO Value (PKR)', data=top_3_vendors, palette='crest')

# Add a clear title to the plot
plt.title('Top 3 Vendors by Total Procurement Amount', fontsize=18, fontweight='bold')

# Label the x-axis
plt.xlabel('Vendor Name', fontsize=14)

# Label the y-axis and format it to prevent scientific notation for currency
plt.ylabel('Total Procurement Amount (PKR)', fontsize=14)
plt.ticklabel_format(style='plain', axis='y') # Ensure full numbers for y-axis ticks

# Rotate x-axis labels if they overlap, and align them to the right
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# Add a grid for better readability of values
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Automatically adjust plot parameters for a tight layout
plt.tight_layout()

# Display the plot
plt.show()

NameError: name 'top_3_vendors' is not defined

<Figure size 1200x700 with 0 Axes>

## Step 6: Final Summary Table

This table provides a concise overview of the top 3 vendors and their respective total procurement amounts, allowing for quick reference to the key findings.

In [23]:
# Display the final summary table of the top 3 vendors
print("Summary Table: Top 3 Vendors by Total Procurement")
display(top_3_vendors.style.format({"PO Value (PKR)": "PKR {:,.2f}"})) # Format as currency for better readability

Summary Table: Top 3 Vendors by Total Procurement


NameError: name 'top_3_vendors' is not defined